In [ ]:
%load_ext autoreload
%autoreload 2

# Import custom module_scripts
from automated_scripts import nemo_0_load_img, nemo_8_interlayer_nematic, nemo_morph_curvature, nemo_morph_thickness
from module_scripts import analysis, datahandler, visuals

# Import Python essentials
import os
import numpy as np
import seaborn as sns

# Standard plotting parameters
channel_colours = {
    0: ["Greens_r", "Greens", "green"],
    1: ["inferno", "inferno_r", "inferno"],
    2: ["Blues_r", "Blues", "blue"],
    3: ["Reds_r", "Reds", "red"],
}
# Initialise Napari viewer once for faster visualisation afterwards
visuals.view_mesh([])

# Simulated

In [ ]:
expected_defects = datahandler.load_array(
    folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo_figure_runs/',
    name="simulated_vesicle")
found_defects = datahandler.load_array(
    folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo_figure_runs/simulated_vesicle/t=0_c=0/data/inner_mesh_smooth_subset_proj_0.0_to_5.0_um_mean/',
    name="defect_coords")
dist_all = []
for d_f in found_defects:
    for d_e in expected_defects:
        dist = np.linalg.norm(d_f - d_e)
        if dist < 100:
            dist_all.append(dist)

sns.boxplot(dist_all)

In [ ]:
found_defect_rel_dists = datahandler.load_array(
    folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo_figure_runs/simulated_vesicle/t=0_c=0/data/inner_mesh_smooth_subset_proj_0.0_to_5.0_um_mean/',
    name="defect_rel-dists")[:, -1]
# sns.boxplot(found_defect_rel_dists, showfliers=True)
sns.histplot(found_defect_rel_dists)

In [ ]:
pts_unit = ((expected_defects[:, ::-1] - 512 // 2) / 200)
unique_arc_lengths = 200 * np.arccos(
    np.clip(np.dot(pts_unit, pts_unit.T)[np.triu_indices(len(expected_defects), k=1)], -1.0, 1.0))
import numpy as np
import scipy.stats as stats

# 1. Calculate Descriptive Statistics
stats_found = {
    'mean': np.mean(found_defect_rel_dists),
    'std': np.std(found_defect_rel_dists),
    'median': np.median(found_defect_rel_dists)
}
stats_expected = {
    'mean': np.mean(unique_arc_lengths),
    'std': np.std(unique_arc_lengths),
    'median': np.median(unique_arc_lengths)
}

# 2. Mean Absolute Error (MAE)
mae = np.mean(np.abs(found_defect_rel_dists - unique_arc_lengths))

# 3. Statistical Significance Tests
# Paired t-test (assumes the pairs in both arrays match 1:1 in order)
t_stat, t_pval = stats.ttest_rel(found_defect_rel_dists, unique_arc_lengths)

# Kolmogorov-Smirnov test (compares shape/distribution independent of ordering)
ks_stat, ks_pval = stats.ks_2samp(found_defect_rel_dists, unique_arc_lengths)

print(
    f"Found    -> Mean: {stats_found['mean']:.2f}, Std: {stats_found['std']:.2f}, Median: {stats_found['median']:.2f}")
print(
    f"Expected -> Mean: {stats_expected['mean']:.2f}, Std: {stats_expected['std']:.2f}, Median: {stats_expected['median']:.2f}")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Paired t-test: p-value = {t_pval:.4f} (t = {t_stat:.2f})")
print(f"KS test (Distribution): p-value = {ks_pval:.4f} (KS stat = {ks_stat:.2f})")

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, euclidean

# Assuming found_defects is already defined as your Nx3 array
# found_defects = ...

# 1. Histogram of Unique Pairwise Distances
plt.figure()
unique_pairwise_distances = pdist(found_defects, metric='euclidean')
sns.histplot(unique_pairwise_distances)
plt.xlabel('Distance')
plt.title('Histogram of Unique Pairwise Distances')

# 2. 3D Plot with Connection Lines Coloured by Length
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Plot points
ax.scatter(found_defects[:, 0], found_defects[:, 1], found_defects[:, 2], color='red', s=50, zorder=5)

# Setup colormap based on the distance range
cmap = plt.cm.viridis
norm = plt.Normalize(vmin=unique_pairwise_distances.min(), vmax=unique_pairwise_distances.max())

# Plot lines between all pairs
N = len(found_defects)
for i in range(N):
    for j in range(i + 1, N):
        dist = euclidean(found_defects[i], found_defects[j])
        line_color = cmap(norm(dist))

        ax.plot(
            [found_defects[i, 0], found_defects[j, 0]],
            [found_defects[i, 1], found_defects[j, 1]],
            [found_defects[i, 2], found_defects[j, 2]],
            color=line_color, alpha=0.6
        )

# Add colorbar to show distance scale
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label='Distance')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Pairwise Connections (Coloured by Length)')

plt.show()

In [ ]:
# ==== Choose image ====
# [!] WINDOWS: Sometimes the r before the file path string is needed, no idea why.
img_path = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo_figure_runs/simulated_vesicle.tif'

# ==== Load image ====
t_select = 0
c_select = 0

img_load = nemo_0_load_img.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                img_slice_colour=channel_colours[c_select][0],
                                img_maxproj_colour=channel_colours[c_select][1],
                                render_colour=channel_colours[c_select][2],
                                render=False, show_figures=True)
resdata_dir, resfig_dir = datahandler.create_resdirs(img_path, ct_label=f"t={t_select}_c={c_select}")
img_raw, img_dim, img_scale, img_unit = img_load
print(f'Found meshes: {[i.split(".ply")[0] for i in os.listdir(resdata_dir) if i.endswith(".ply")]}')

In [ ]:
# ==== 3D render result ====
mesh_to_overlay_names = ["inner_mesh_smooth_subset"]
meshes_to_overlay = [datahandler.load_mesh(filepath=os.path.join(resdata_dir, f"{mesh_to_overlay_name}.ply")) for
                     mesh_to_overlay_name in mesh_to_overlay_names]
visuals.view_mesh(mesh_list=meshes_to_overlay, mesh_titles=mesh_to_overlay_names,
                  mesh_colors=["white"] * len(meshes_to_overlay), mesh_shadings=["flat"] * len(meshes_to_overlay),
                  img=img_raw, mesh_opacities=[0.9] * len(meshes_to_overlay), scale=img_scale, vec_freq=50,
                  hide_vectors=False, vec_length=10, vec_edge_width=0.2)

In [ ]:
# ==== 3D render result ====
layer_label = 'inner_mesh_smooth_subset_proj_0.0_to_5.0_um_mean'
nematic_avg_label = 'r-50.0um'
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
s_2dcurv = datahandler.load_array(name=f"S-order_2dcurved_{nematic_avg_label}",
                                  folderpath=resdata_dir_layer)
directors_2dcurved_avg = datahandler.load_array(name=f"directors-avg_2dcurved_{nematic_avg_label}",
                                                folderpath=resdata_dir_layer)
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
defect_idxs_calc = datahandler.load_array(name=f"top-charge_2dcurved_idxs",
                                          folderpath=resdata_dir_layer).astype(int)
m_charge = datahandler.load_array(name=f"top-charge_2dcurved",
                                  folderpath=resdata_dir_layer)
charge_pol_linked_idxs = datahandler.load_array(name=f"def-pol_2dcurved_idxs_expanded",
                                                folderpath=resdata_dir_layer).astype(int)
pol_vecfield = datahandler.load_array(name=f"def-pol_2dcurved",
                                      folderpath=resdata_dir_layer)
tan_x = datahandler.load_array("tan_x", folderpath=resdata_dir_layer)
tan_y = datahandler.load_array("tan_y", folderpath=resdata_dir_layer)
normals = layer_mesh.vertex_normals[idxs_sel]

directors_2dcurved = datahandler.load_array(name=f"directors_2dcurved",
                                            folderpath=resdata_dir_layer)
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved, mesh_shading="flat",
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys_r"),
                                    vec_colors="red", img=img_raw, vec_length=10, vec_edge_width=0.7,
                                    scale=img_scale, center_marker_vector=False)

visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="flat",
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys_r"),
                                    vec_colors=visuals.color_scalar(s_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]], vec_edge_width=0.7,
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400,
                                    marker_vectors=pol_vecfield if len(pol_vecfield) > 0 else None,
                                    marker_vectors_length=50, vec_length=10,
                                    marker_vector_width=3,
                                    marker_vectors_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs],
                                                                              manual_vminmax=[-1, 1],
                                                                              cmap="rainbow") if len(
                                        charge_pol_linked_idxs) > 0 else None, img=img_raw,
                                    scale=img_scale, center_marker_vector=False)

In [ ]:
defect_idxs_calc

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="flat",
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys_r"),
                                    vec_colors=visuals.color_scalar(s_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]][[0]], vec_edge_width=0.7,
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow")[[0]], marker_size=400,
                                    marker_vectors=pol_vecfield if len(pol_vecfield) > 0 else None,
                                    marker_vectors_length=50, vec_length=10,
                                    marker_vector_width=3,
                                    marker_vectors_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs],
                                                                              manual_vminmax=[-1, 1],
                                                                              cmap="rainbow") if len(
                                        charge_pol_linked_idxs) > 0 else None, img=img_raw,
                                    scale=img_scale, center_marker_vector=False)

In [ ]:
debug_layer_label = 'TEST'
debug_resdata_dir_layer = os.path.join(resdata_dir, debug_layer_label)
debug_resfig_dir_layer = os.path.join(resfig_dir, debug_layer_label)
debug_proj_layer = datahandler.load_array("intensities", folderpath=debug_resdata_dir_layer)
debug_layer_mesh = datahandler.load_mesh(os.path.join(debug_resdata_dir_layer, "layer_mesh.ply"))
debug_tan_x = datahandler.load_array("tan_x", folderpath=debug_resdata_dir_layer)
debug_tan_y = datahandler.load_array("tan_y", folderpath=debug_resdata_dir_layer)
debug_idxs_sel = datahandler.load_array("calcindeces", folderpath=debug_resdata_dir_layer).astype(int)
debug_directors_2dcurved_avg = datahandler.load_array(name=f"debug-idx=0_directors_2dcurved",
                                                      folderpath=debug_resdata_dir_layer)
debug_normal = debug_layer_mesh.vertex_normals[debug_idxs_sel]

#theta = -44.33
visuals.view_colored_mesh_dir_field(mesh=debug_layer_mesh,
                                    mesh_vert_colors=visuals.color_scalar(debug_proj_layer, normalise=True,
                                                                          cmap="Greys_r"),
                                    center_vectors=False, vector_style="arrow",
                                    vec_colors=["orange", "lightblue", "white"],
                                    directors=np.concatenate(
                                        (np.column_stack((debug_directors_2dcurved_avg[:, :3], debug_tan_x)),
                                         np.column_stack((debug_directors_2dcurved_avg[:, :3], debug_tan_y)),
                                         np.column_stack((debug_directors_2dcurved_avg[:, :3], debug_normal))),
                                        axis=0), vec_edge_width=3,
                                    vec_length=30, marker_vectors=debug_directors_2dcurved_avg,
                                    marker_vectors_length=30, marker_vectors_style="line", marker_vector_width=4,
                                    center_marker_vector=True)

In [ ]:
# ==== Visualise averaging patch ====
idxs_neigh = np.load(os.path.join(resdata_dir_layer, f"{nematic_avg_label}_idxs_neigh.npz"), allow_pickle=True)[
    "idxs_neigh"]
patch_sel_idx = 5738
print(patch_sel_idx)
patch_color = np.ones(len(directors_2dcurved_avg))
patch_color[idxs_neigh[patch_sel_idx]] = 0.5
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved, mesh_shading="flat",
                                    mesh_vert_colors="grey",
                                    vec_colors=visuals.color_scalar(patch_color, cmap="bwr"),
                                    vec_length=10, vec_edge_width=0.7,
                                    markers=layer_mesh.vertices[idxs_sel[idxs_neigh[patch_sel_idx][0]]],
                                    marker_colors="white", marker_size=10)

In [ ]:
# # ==== Extract nematic ====
# layer_label = "TEST"
# nematic_patch_mode = "radius"
# nematic_patch_size = 20
# nematic_compute_num = 1
# nematic_normal_validity_k = 20
# nematic_normal_validity_thresh = 0.99
# nematic_grid_n = 30
# nematic_extracted = nemo_4_extract_nematic.main(img_path=img_path,
#                                                 t_select=t_select, c_select=c_select,
#                                                 layer_label=layer_label,
#                                                 patch_mode=nematic_patch_mode, patch_size=nematic_patch_size,
#                                                 compute_num=nematic_compute_num,
#                                                 normal_validity_k=nematic_normal_validity_k,
#                                                 normal_validity_thresh=nematic_normal_validity_thresh,
#                                                 grid_n_2dcurve_analysis=nematic_grid_n,
#                                                 debug_2dcurve_analysis=True, show_figures=True, render=False)
# # layer_label, layer_mesh, proj_layer, idxs_neigh, idxs_sel, directors_2dcurved = nematic_extracted

# Hydra Foot

In [ ]:
# ==== Choose image ====
# [!] WINDOWS: Sometimes the r before the file path string is needed, no idea why.
img_path = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo_figure_runs/hydra-foot.tif'

# ==== Load image ====
t_select = 0
c_select = 0

img_load = nemo_0_load_img.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                img_slice_colour=channel_colours[c_select][0],
                                img_maxproj_colour=channel_colours[c_select][1],
                                render_colour=channel_colours[c_select][2],
                                render=False, show_figures=True)
resdata_dir, resfig_dir = datahandler.create_resdirs(img_path, ct_label=f"t={t_select}_c={c_select}")
img_raw, img_dim, img_scale, img_unit = img_load

In [ ]:
print(f'Found meshes: {[i.split(".ply")[0] for i in os.listdir(resdata_dir) if i.endswith(".ply")]}')
# ==== 3D render multiple layers ====
layer_label_list = [i for i in os.listdir(resdata_dir) if os.path.isdir(os.path.join(resdata_dir, i))]
print(f"Rendering {layer_label_list} ...")
# layer_label_list = ["sampling_mesh_proj_9.0_to_11.0_um_mean"]
layer_mesh_list = []
layer_projection_list = []
for layer_label_i in layer_label_list:
    resdata_dir_layer = os.path.join(resdata_dir, layer_label_i)
    resfig_dir_layer = os.path.join(resfig_dir, layer_label_i)
    proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
    layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
    if layer_mesh is None:
        print(f"[!] Could not find mesh for layer {layer_label_i} !")
        break
    layer_mesh_list.append(layer_mesh)
    layer_projection_list.append(visuals.color_scalar(proj_layer, normalise=True, cmap="inferno"))
visuals.view_colored_mesh_multiple(mesh_list=layer_mesh_list,
                                   mesh_blending_list=["opaque" for _ in range(len(layer_mesh_list))],
                                   name_list=layer_label_list,
                                   vert_colors_list=layer_projection_list, img=img_raw, scale=img_scale)

In [ ]:
# ==== 3D render result ====
mesh_to_overlay_names = ["sampling_bottom", "sampling_top"]
meshes_to_overlay = [datahandler.load_mesh(filepath=os.path.join(resdata_dir, f"{mesh_to_overlay_name}.ply")) for
                     mesh_to_overlay_name in mesh_to_overlay_names]
visuals.view_mesh(mesh_list=meshes_to_overlay, mesh_titles=mesh_to_overlay_names,
                  mesh_colors=["white"] * len(meshes_to_overlay), mesh_shadings=["flat"] * len(meshes_to_overlay),
                  img=img_raw, mesh_opacities=[0.9] * len(meshes_to_overlay), scale=img_scale, vec_freq=100,
                  hide_vectors=True, vec_length=10, vec_edge_width=0.2)

In [ ]:
# ==== Find available nematic analyses ====
found_projected_layers = [i for i in os.listdir(resdata_dir) if os.path.isdir(os.path.join(resdata_dir, i))]
found_analysed_layers = [
    (layer, f.split("_")[-1].split(".csv")[0])
    for layer in found_projected_layers
    for f in os.listdir(os.path.join(resdata_dir, layer))
    if f.startswith("S-order_2dcurved") and f.endswith(".csv")
]
print(found_analysed_layers)

In [ ]:
# ==== 3D render result ====
for layer_label in ['sampling_bottom_proj_-0.6_to_-0.4_um_max', 'sampling_bottom_proj_4.4_to_4.6_um_max']:
    nematic_avg_label = 'r-50.0um'
    resdata_dir_layer = os.path.join(resdata_dir, layer_label)
    resfig_dir_layer = os.path.join(resfig_dir, layer_label)
    proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
    layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
    directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)
    idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
    s_2dcurv = datahandler.load_array(name=f"S-order_2dcurved_{nematic_avg_label}",
                                      folderpath=resdata_dir_layer)
    directors_2dcurved_avg = datahandler.load_array(name=f"directors-avg_2dcurved_{nematic_avg_label}",
                                                    folderpath=resdata_dir_layer)
    directors_2dcurved = datahandler.load_array(name=f"directors_2dcurved",
                                                folderpath=resdata_dir_layer)
    vec_length = 7
    vec_edge_width = 0.4
    visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved,
                                        vec_colors="red",
                                        mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                              cmap="Greys_r"),
                                        vec_length=vec_length, vec_edge_width=vec_edge_width)
    visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved,
                                        vec_colors=visuals.color_scalar(s_2dcurv, manual_vminmax=[0, 1]),
                                        mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                              cmap="Greys_r"),
                                        vec_length=vec_length, vec_edge_width=vec_edge_width)

    visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                        vec_colors=visuals.color_scalar(s_2dcurv, manual_vminmax=[0, 1]),
                                        mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                              cmap="Greys_r"),
                                        vec_length=vec_length, vec_edge_width=vec_edge_width, img=img_raw,
                                        scale=img_scale)

In [ ]:
# ==== Visualise averaging patch ====
idxs_neigh = np.load(os.path.join(resdata_dir_layer, f"{nematic_avg_label}_idxs_neigh.npz"), allow_pickle=True)[
    "idxs_neigh"]
patch_sel_idx = 5738
print(patch_sel_idx)
patch_color = np.ones(len(directors_2dcurved_avg))
patch_color[idxs_neigh[patch_sel_idx]] = 0.5
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved, mesh_shading="flat",
                                    mesh_vert_colors="grey",
                                    vec_colors=visuals.color_scalar(patch_color, cmap="bwr"),
                                    vec_length=10, vec_edge_width=0.7,
                                    markers=layer_mesh.vertices[idxs_sel[idxs_neigh[patch_sel_idx][0]]],
                                    marker_colors="white", marker_size=10)

In [ ]:
# ==== Analyse inter-layer nematic order ====
inter_layer_s_layer_name_1 = 'sampling_bottom_proj_4.4_to_4.6_um_max'
inter_layer_s_patch_label_1 = 'r-50.0um'
inter_layer_s_layer_name_2 = 'sampling_bottom_proj_-0.6_to_-0.4_um_max'
inter_layer_s_patch_label_2 = 'r-50.0um'
inter_layer_s_analysed = nemo_8_interlayer_nematic.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                                        layer_name_1=inter_layer_s_layer_name_1,
                                                        layer_name_2=inter_layer_s_layer_name_2,
                                                        patch_label_1=inter_layer_s_patch_label_1,
                                                        patch_label_2=inter_layer_s_patch_label_2,
                                                        show_figures=True, render=False)
layer_mesh_1, layer_mesh_2, proj_layer_1, proj_layer_2, field_1, field_2, plotted_vecfields, s_2dcurv_layer_1, s_2dcurv_layer_2, inter_layer_s = inter_layer_s_analysed

In [ ]:
# ==== 3D render results ====
if inter_layer_s_analysed is not None:
    visuals.view_colored_mesh_dir_field(mesh=layer_mesh_1, directors=plotted_vecfields,
                                        vec_colors=np.concatenate((["grey"] * len(field_2),
                                                                   visuals.color_scalar(inter_layer_s,
                                                                                        manual_vminmax=[0, 1],
                                                                                        cmap="Spectral")), axis=0),
                                        mesh_vert_colors="black",
                                        vec_edge_width=0.3, vec_length=7)

In [ ]:
# ==== 3D render results ====
if inter_layer_s_analysed is not None:
    visuals.view_colored_mesh_dir_field(mesh=layer_mesh_1, directors=plotted_vecfields,
                                        vec_colors=np.concatenate((["grey"] * len(field_2),
                                                                   visuals.color_scalar(inter_layer_s,
                                                                                        manual_vminmax=[0, 1],
                                                                                        cmap="Spectral")), axis=0),
                                        mesh_vert_colors="black",
                                        vec_edge_width=0.3, vec_length=7)

In [ ]:
# ==== Calculate thicknesses ====
thickness_mesh_1_name = "sampling_bottom"
thickness_mesh_2_name = "sampling_top"
# thickness_num_samples = 3000
# thickness_interp_k = 10
# mesh_1, mesh_2, full_dist_vals = nemo_morph_thickness.main(img_path=img_path, t_select=t_select, c_select=c_select,
#                                                            mesh_1_name=thickness_mesh_1_name,
#                                                            mesh_2_name=thickness_mesh_2_name,
#                                                            thickness_sampl_number=thickness_num_samples,
#                                                            interp_k=thickness_interp_k, show_figures=True, render=False)
# ==== 3D Render result ====
mesh_1 = datahandler.load_mesh(os.path.join(resdata_dir, f"{thickness_mesh_1_name}.ply"))
mesh_2 = datahandler.load_mesh(os.path.join(resdata_dir, f"{thickness_mesh_2_name}.ply"))
full_dist_vals = datahandler.load_array(name=f"{thickness_mesh_1_name}_VS_{thickness_mesh_2_name}_thickness",
                                        folderpath=resdata_dir)
visuals.view_colored_mesh_multiple([mesh_1, mesh_2],
                                   [visuals.color_scalar(full_dist_vals, normalise=True, cmap="coolwarm"),
                                    "white"], mesh_blending_list=["opaque", "translucent"],
                                   mesh_opacity_list=[1.0, 0.3], img=img_raw, scale=img_scale)

In [ ]:
tan_x, tan_y = analysis.create_tangential_basis(
    mesh=datahandler.load_mesh(filepath=os.path.join(resdata_dir, f"sampling_bottom.ply")))

In [ ]:
# ==== Calculate curvatures ====
curv_mesh_name = 'sampling_bottom'
curv_num_samples = 2000
curv_radius = 20
curv_interp_k = 10
curv_flip_normals = False
# mesh_curv, full_C_gauss, full_C_mean = nemo_morph_curvature.main(img_path=img_path, t_select=t_select,
#                                                                  c_select=c_select,custom_basis=(tan_x, tan_y),
#                                                                  num_samples=curv_num_samples, radius=curv_radius,
#                                                                  interp_k=curv_interp_k,
#                                                                  mesh_name=curv_mesh_name,
#                                                                  flip_normals=curv_flip_normals,
#                                                                  show_figures=True, render=False, plot_maxprojections=True)
# ==== 3D render result ====
mesh_curv = datahandler.load_mesh(os.path.join(resdata_dir, f"{curv_mesh_name}.ply"))
full_C_gauss = datahandler.load_array(f"{curv_mesh_name}_gauss_curv_r-{curv_radius}{img_unit}",
                                      folderpath=resdata_dir)
full_C_mean = datahandler.load_array(f"{curv_mesh_name}_mean_curv_r-{curv_radius}{img_unit}",
                                     folderpath=resdata_dir)
visuals.view_colored_mesh_multiple([mesh_curv, mesh_curv],
                                   [visuals.color_scalar(full_C_gauss, manual_vminmax=[
                                       -max(abs(full_C_gauss.min()), abs(full_C_gauss.max())),
                                       max(abs(full_C_gauss.min()), abs(full_C_gauss.max()))], cmap="coolwarm"),
                                    visuals.color_scalar(full_C_mean, normalise=True, cmap="Spectral")],
                                   name_list=[f"Gauss {curv_mesh_name}", f"Mean {curv_mesh_name}"], img=img_raw,
                                   scale=img_scale)

# Hydra Head

In [ ]:
# ==== Choose image ====
# [!] WINDOWS: Sometimes the r before the file path string is needed, no idea why.
img_path = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo_figure_runs/hydra-head.tif'

# ==== Load image ====
t_select = 0
c_select = 0

img_load = nemo_0_load_img.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                img_slice_colour=channel_colours[c_select][0],
                                img_maxproj_colour=channel_colours[c_select][1],
                                render_colour=channel_colours[c_select][2],
                                render=False, show_figures=True)
resdata_dir, resfig_dir = datahandler.create_resdirs(img_path, ct_label=f"t={t_select}_c={c_select}")
img_raw, img_dim, img_scale, img_unit = img_load
print(f'Found meshes: {[i.split(".ply")[0] for i in os.listdir(resdata_dir) if i.endswith(".ply")]}')

In [ ]:

# ==== 3D render multiple layers ====
layer_label_list = [i for i in os.listdir(resdata_dir) if os.path.isdir(os.path.join(resdata_dir, i))]
print(f"Rendering {layer_label_list} ...")
# layer_label_list = ["sampling_mesh_proj_9.0_to_11.0_um_mean"]
layer_mesh_list = []
layer_projection_list = []
for layer_label_i in layer_label_list:
    resdata_dir_layer = os.path.join(resdata_dir, layer_label_i)
    resfig_dir_layer = os.path.join(resfig_dir, layer_label_i)
    proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
    layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
    if layer_mesh is None:
        print(f"[!] Could not find mesh for layer {layer_label_i} !")
        break
    layer_mesh_list.append(layer_mesh)
    layer_projection_list.append(visuals.color_scalar(proj_layer, normalise=True, cmap="inferno"))
visuals.view_colored_mesh_multiple(mesh_list=layer_mesh_list,
                                   mesh_blending_list=["opaque" for _ in range(len(layer_mesh_list))],
                                   name_list=layer_label_list,
                                   vert_colors_list=layer_projection_list, img=img_raw, scale=img_scale)

In [ ]:
# ==== Find available nematic analyses ====
found_projected_layers = [i for i in os.listdir(resdata_dir) if os.path.isdir(os.path.join(resdata_dir, i))]
found_analysed_layers = [
    (layer, f.split("_")[-1].split(".csv")[0])
    for layer in found_projected_layers
    for f in os.listdir(os.path.join(resdata_dir, layer))
    if f.startswith("S-order_2dcurved") and f.endswith(".csv")
]
print(found_analysed_layers)

In [ ]:
# ==== 3D render result ====
layer_label = 'sampling_mesh_proj_27.9_to_28.1_um_mean'
nematic_avg_label = 'r-30.0um'
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
s_2dcurv = datahandler.load_array(name=f"S-order_2dcurved_{nematic_avg_label}",
                                  folderpath=resdata_dir_layer)
directors_2dcurved_avg = datahandler.load_array(name=f"directors-avg_2dcurved_{nematic_avg_label}",
                                                folderpath=resdata_dir_layer)
vec_length = 10
vec_edge_width = 1.0
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(s_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width, img=img_raw, scale=img_scale)

In [ ]:
# ==== Find analysed layers ====
found_analysed_layers = [
    (layer, f.split("_")[-1].split(".csv")[0])
    for layer in [i for i in os.listdir(resdata_dir) if os.path.isdir(os.path.join(resdata_dir, i))]
    for f in os.listdir(os.path.join(resdata_dir, layer))
    if f.startswith("S-order_2dcurved") and f.endswith(".csv")
]
print(found_analysed_layers)

In [ ]:
# ==== Calculate thicknesses ====
thickness_mesh_1_name = "sampling_bottom"
thickness_mesh_2_name = "sampling_top"
thickness_num_samples = 3000
thickness_interp_k = 10
mesh_1, mesh_2, full_dist_vals = nemo_morph_thickness.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                                           mesh_1_name=thickness_mesh_1_name,
                                                           mesh_2_name=thickness_mesh_2_name,
                                                           thickness_sampl_number=thickness_num_samples,
                                                           interp_k=thickness_interp_k, show_figures=True, render=False)
# ==== 3D Render result ====
mesh_1 = datahandler.load_mesh(os.path.join(resdata_dir, f"{thickness_mesh_1_name}.ply"))
mesh_2 = datahandler.load_mesh(os.path.join(resdata_dir, f"{thickness_mesh_2_name}.ply"))
full_dist_vals = datahandler.load_array(name=f"{thickness_mesh_1_name}_VS_{thickness_mesh_2_name}_thickness",
                                        folderpath=resdata_dir)
visuals.view_colored_mesh_multiple([mesh_1, mesh_2],
                                   [visuals.color_scalar(full_dist_vals, normalise=True, cmap="coolwarm"),
                                    "white"], mesh_blending_list=["opaque", "translucent"],
                                   mesh_opacity_list=[1.0, 0.3], img=img_raw, scale=img_scale)

In [ ]:
mesh = datahandler.load_mesh(filepath=os.path.join(resdata_dir, f"sampling_mesh.ply"))
mesh.invert()
tan_x, tan_y = analysis.create_tangential_basis(
    mesh=mesh)

In [ ]:
# ==== Calculate curvatures ====
curv_mesh_name = 'sampling_mesh'
curv_num_samples = 8000
curv_radius = 40
curv_interp_k = 20
curv_flip_normals = True
mesh_curv, full_C_gauss, full_C_mean = nemo_morph_curvature.main(img_path=img_path, t_select=t_select,
                                                                 c_select=c_select,
                                                                 num_samples=curv_num_samples, radius=curv_radius,
                                                                 interp_k=curv_interp_k,
                                                                 mesh_name=curv_mesh_name, custom_basis=(tan_x, tan_y),
                                                                 flip_normals=curv_flip_normals,
                                                                 plot_maxprojections=True,
                                                                 show_figures=True, render=False)
# ==== 3D render result ====
mesh_curv = datahandler.load_mesh(os.path.join(resdata_dir, f"{curv_mesh_name}.ply"))
full_C_gauss = datahandler.load_array(f"{curv_mesh_name}_gauss_curv_r-{curv_radius}{img_unit}",
                                      folderpath=resdata_dir)
full_C_mean = datahandler.load_array(f"{curv_mesh_name}_mean_curv_r-{curv_radius}{img_unit}",
                                     folderpath=resdata_dir)

In [ ]:
visuals.view_colored_mesh_multiple([mesh_curv, mesh_curv],
                                   [visuals.color_scalar(full_C_gauss, manual_vminmax=[
                                       -max(abs(full_C_gauss.min()), abs(full_C_gauss.max())),
                                       max(abs(full_C_gauss.min()), abs(full_C_gauss.max()))], cmap="coolwarm"),
                                    visuals.color_scalar(full_C_mean, normalise=True, cmap="Spectral")],
                                   name_list=[f"Gauss {curv_mesh_name}", f"Mean {curv_mesh_name}"], img=img_raw,
                                   scale=img_scale)

# Tangential Basis

In [ ]:
full_tan_x, full_tan_y = analysis.create_tangential_basis(mesh=layer_mesh)
full_normals = layer_mesh.vertex_normals
freq = 50
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, mesh_vert_colors="white",
                                    center_vectors=False, vector_style="arrow",
                                    vec_colors=np.concatenate(
                                        (["orange"] * len(full_tan_x[::freq]), ["lightblue"] * len(full_tan_y[::freq]),
                                         ["grey"] * len(full_normals[::freq])),
                                        axis=0),
                                    directors=np.concatenate(
                                        (np.column_stack((layer_mesh.vertices, full_tan_x))[::freq],
                                         np.column_stack((layer_mesh.vertices, full_tan_y))[::freq],
                                         np.column_stack(
                                             (layer_mesh.vertices, full_normals))[::freq]),
                                        axis=0), vec_edge_width=0.2,
                                    vec_length=20)